## Langfuse Cloud Connection

Initialize the Langfuse client and verify that the application can connect to the cloud observability platform.

In [1]:
import os
from dotenv import load_dotenv
from langfuse import get_client

load_dotenv()

langfuse = get_client()

print("Public key configured:", bool(os.getenv("LANGFUSE_PUBLIC_KEY")))
print("Secret key configured:", bool(os.getenv("LANGFUSE_SECRET_KEY")))
print("Authenticated:", langfuse.auth_check())

Public key configured: True
Secret key configured: True
Authenticated: True


## 1. Inspect Existing Vector Indexes

Inspect the Discovery and Detail Chroma collections to understand their document counts, collection configuration, IDs and metadata structure.

This establishes the baseline before implementing indexing CRUD operations.

In [4]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

GOLDEN_PATH = (
    PROJECT_DIR
    / "notebooks/golden_dataset/discovery/seva_discovery_golden_v1.jsonl"
)

DISCOVERY_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\discovery")

DETAIL_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\detail")

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [5]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 120.34 seconds
Embedding dimensions: 384
Model loaded successfully!


In [6]:
from collections import defaultdict
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

VECTORSTORE_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores")
golden_df = pd.read_json(GOLDEN_PATH, lines=True)

discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [7]:
import time
from langfuse import get_client

langfuse = get_client()

with langfuse.start_as_current_observation(
    name="indexing.inspect",
    as_type="span"
) as span:

    start = time.perf_counter()

    discovery_collection = discovery_db._collection
    detail_collection = detail_db._collection

    discovery_count = discovery_collection.count()
    detail_count = detail_collection.count()

    discovery_sample = discovery_collection.get(
        limit=3,
        include=["metadatas", "documents"]
    )

    detail_sample = detail_collection.get(
        limit=3,
        include=["metadatas", "documents"]
    )

    latency = time.perf_counter() - start

    span.update(
        metadata={
            "discovery_count": discovery_count,
            "detail_count": detail_count,
            "sample_size": 3
        },
        level="DEFAULT"
    )

print("DISCOVERY COLLECTION")
print("Name:", discovery_collection.name)
print("Count:", discovery_count)
print("Metadata:", discovery_collection.metadata)
print("Sample IDs:", discovery_sample["ids"])
print("Sample metadata:", discovery_sample["metadatas"])

print("\nDETAIL COLLECTION")
print("Name:", detail_collection.name)
print("Count:", detail_count)
print("Metadata:", detail_collection.metadata)
print("Sample IDs:", detail_sample["ids"])
print("Sample metadata:", detail_sample["metadatas"])

print(f"\nInspection latency: {latency:.4f} seconds")

langfuse.flush()

DISCOVERY COLLECTION
Name: seva_discovery_v1
Count: 23438
Metadata: None
Sample IDs: ['LBezF5gB5uF80_MGGY81_OVERVIEW_001_CHUNK_001', 'LBezF5gB5uF80_MGGY81_BENEFITS_001_CHUNK_001', 'LBezF5gB5uF80_MGGY81_BENEFICIARIES_001_CHUNK_001']
Sample metadata: [{'child_type': 'overview', 'scheme_id': 'LBezF5gB5uF80_MGGY81', 'level': 'Central', 'language': 'en', 'states': '["All"]', 'chunk_index': 1, 'categories': '["Business & Entrepreneurship", "Banking,Financial Services and Insurance", "Social welfare & Empowerment"]', 'child_id': 'LBezF5gB5uF80_MGGY81_OVERVIEW_001', 'chunk_id': 'LBezF5gB5uF80_MGGY81_OVERVIEW_001_CHUNK_001', 'source_field': 'brief_description', 'parent_id': 'LBezF5gB5uF80_MGGY81_OVERVIEW_001'}, {'scheme_id': 'LBezF5gB5uF80_MGGY81', 'categories': '["Business & Entrepreneurship", "Banking,Financial Services and Insurance", "Social welfare & Empowerment"]', 'level': 'Central', 'chunk_index': 1, 'source_field': 'benefits', 'parent_id': 'LBezF5gB5uF80_MGGY81_BENEFITS_001', 'states':

## 2. Validate Document IDs and Relationships

Analyze the metadata of both Chroma collections to validate vector IDs, scheme IDs, parent-child relationships and chunk distribution.

This helps establish safe identifiers for insertion, update and deletion operations.

In [7]:
import pandas as pd
import time

BATCH_SIZE = 500

def load_collection_metadata(collection):
    records = []
    total = collection.count()

    for offset in range(0, total, BATCH_SIZE):
        batch = collection.get(
            limit=BATCH_SIZE,
            offset=offset,
            include=["metadatas"]
        )

        for vector_id, metadata in zip(
            batch["ids"], batch["metadatas"]
        ):
            records.append({
                "vector_id": vector_id,
                **(metadata or {})
            })

    return pd.DataFrame(records)


with langfuse.start_as_current_observation(
    name="indexing.validate_relationships",
    as_type="span"
) as span:

    start = time.perf_counter()

    collection_data = {
        "discovery": load_collection_metadata(discovery_collection),
        "detail": load_collection_metadata(detail_collection)
    }

    required = ["scheme_id", "parent_id", "child_id", "chunk_id"]

    for name, df in collection_data.items():

        for field in required:
            if field not in df.columns:
                df[field] = None

        df["id_matches_chunk_id"] = (
            df["vector_id"] == df["chunk_id"]
        )

        print(f"\n{name.upper()}")
        print("Total chunks:", len(df))
        print("Unique schemes:", df["scheme_id"].nunique())
        print("Unique parents:", df["parent_id"].nunique())
        print("Unique children:", df["child_id"].nunique())

        print("Missing metadata:")
        print(df[required].isna().sum().to_dict())

        print(
            "Vector ID mismatches:",
            (~df["id_matches_chunk_id"]).sum()
        )

        print("\nChunks per child:")
        print(df.groupby("child_id").size().describe().round(2))

    discovery_schemes = set(
        collection_data["discovery"]["scheme_id"].dropna()
    )

    detail_schemes = set(
        collection_data["detail"]["scheme_id"].dropna()
    )

    overlap = discovery_schemes & detail_schemes

    latency = time.perf_counter() - start

    span.update(
        metadata={
            "discovery_schemes": len(discovery_schemes),
            "detail_schemes": len(detail_schemes),
            "shared_schemes": len(overlap),
            "batch_size": BATCH_SIZE
        }
    )

print("\nCROSS-COLLECTION ANALYSIS")
print("Discovery schemes:", len(discovery_schemes))
print("Detail schemes:", len(detail_schemes))
print("Shared schemes:", len(overlap))
print(f"\nValidation latency: {latency:.4f} seconds")

langfuse.flush()


DISCOVERY
Total chunks: 23438
Unique schemes: 4428
Unique parents: 18393
Unique children: 18393
Missing metadata:
{'scheme_id': 0, 'parent_id': 0, 'child_id': 0, 'chunk_id': 0}
Vector ID mismatches: 0

Chunks per child:
count    18393.00
mean         1.27
std          0.98
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         42.00
dtype: float64

DETAIL
Total chunks: 67945
Unique schemes: 4426
Unique parents: 57973
Unique children: 57973
Missing metadata:
{'scheme_id': 0, 'parent_id': 0, 'child_id': 0, 'chunk_id': 0}
Vector ID mismatches: 0

Chunks per child:
count    57973.00
mean         1.17
std          0.74
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         17.00
dtype: float64

CROSS-COLLECTION ANALYSIS
Discovery schemes: 4428
Detail schemes: 4426
Shared schemes: 4426

Validation latency: 114.7646 seconds


## 3. Investigate Discovery-Only Schemes

Identify schemes present in Discovery but absent from Detail. Inspect their available sections, parent IDs and chunk counts to determine whether they are intentionally overview-only or have incomplete detail indexing.

In [8]:
import time
import pandas as pd

with langfuse.start_as_current_observation(
    name="indexing.investigate_missing_schemes",
    as_type="span"
) as span:

    start = time.perf_counter()

    discovery_df = collection_data["discovery"]
    detail_df = collection_data["detail"]

    discovery_only = sorted(
        set(discovery_df["scheme_id"].dropna())
        - set(detail_df["scheme_id"].dropna())
    )

    missing_scheme_df = discovery_df[
        discovery_df["scheme_id"].isin(discovery_only)
    ].copy()

    summary = (
        missing_scheme_df
        .groupby("scheme_id")
        .agg(
            chunks=("vector_id", "count"),
            parents=("parent_id", "nunique"),
            sections=("child_type", lambda x: sorted(x.dropna().unique().tolist())),
            source_fields=("source_field", lambda x: sorted(x.dropna().unique().tolist()))
        )
        .reset_index()
    )

    latency = time.perf_counter() - start

    span.update(
        metadata={
            "discovery_only_scheme_count": len(discovery_only),
            "affected_chunk_count": len(missing_scheme_df)
        }
    )

print("Discovery-only scheme IDs:")
print(discovery_only)

print("\nScheme-level investigation:")
display(summary)

print(f"\nInvestigation latency: {latency:.4f} seconds")

langfuse.flush()

Discovery-only scheme IDs:
['5aKstZ4BXu-iUKXq-liy', 'DRi4Yp0B5uF80_MGMmWm']

Scheme-level investigation:


,scheme_id,chunks,parents,sections,source_fields
0,5aKstZ4BXu-iUKXq-liy,1,1,[overview],[brief_description]
1,DRi4Yp0B5uF80_MGMmWm,1,1,[overview],[brief_description]



Investigation latency: 1.0189 seconds


## 4. Inspect Embedding and Collection Configuration

Inspect the existing embedding function, vector dimensions and collection configuration. New documents must use the same embedding dimensions and compatible distance metric as the existing collection.

In [9]:
import time

with langfuse.start_as_current_observation(
    name="indexing.inspect_configuration",
    as_type="span"
) as span:

    start = time.perf_counter()

    for name, collection, vectorstore in [
        ("discovery", discovery_collection, discovery_db),
        ("detail", detail_collection, detail_db)
    ]:
        sample = collection.get(
            limit=1,
            include=["embeddings"]
        )

        embedding_dimension = len(sample["embeddings"][0])

        print(f"\n{name.upper()}")
        print("Embedding dimension:", embedding_dimension)
        print("Embedding function:", type(
            getattr(vectorstore, "_embedding_function", None)
        ).__name__)
        print("Collection configuration:", getattr(
            collection, "configuration", None
        ))

    latency = time.perf_counter() - start

    span.update(metadata={
        "inspection_latency_seconds": latency
    })

print(f"\nConfiguration inspection latency: {latency:.4f} seconds")

langfuse.flush()


DISCOVERY
Embedding dimension: 384
Embedding function: HuggingFaceEmbeddings
Collection configuration: {'hnsw': {'space': 'l2', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': None}

DETAIL
Embedding dimension: 384
Embedding function: HuggingFaceEmbeddings
Collection configuration: {'hnsw': {'space': 'l2', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}, 'spann': None, 'embedding_function': None}

Configuration inspection latency: 0.0969 seconds


## 5. Reusable Version Metadata Function

### Purpose
Create a reusable function to attach indexing version metadata to Discovery and Detail documents before inserting them into ChromaDB.

This ensures consistent version tracking across all indexed chunks.

In [10]:
def apply_version_metadata(documents):
    for doc in documents:
        doc.metadata.update(INDEXING_VERSION_METADATA)

    return documents


# Validate using the synthetic document
versioned_docs = apply_version_metadata([test_doc])

print("Total documents:", len(versioned_docs))
print("Version metadata:", versioned_docs[0].metadata)

Total documents: 1
Version metadata: {'scheme_id': 'VERSION_TEST_001', 'chunk_id': 'VERSION_TEST_001_CHUNK_001', 'source_field': 'overview', 'data_version': 'data_v1', 'chunking_version': 'chunk_v1', 'embedding_version': 'embed_v1', 'indexing_version': 'index_v1'}


## 6. Validate Discovery and Detail Documents

### Purpose
Inspect the existing Discovery and Detail document objects before integrating version metadata into the indexing pipeline.

Confirm document counts and metadata structure without modifying the original documents or ChromaDB collections.

In [16]:
print("Discovery chunks:", discovery_db._collection.count())
print("Detail chunks:", detail_db._collection.count())

Discovery chunks: 23438
Detail chunks: 67945


## 7. Inspect Existing Vector Metadata

### Purpose
Inspect one existing record from each collection to understand the current metadata structure before adding version information.

## Secure Incremental Indexing Pipeline

### Purpose

Implement a secure and versioned indexing pipeline for SEVA_AI.

Security:
- Role-based access control using PyCasbin
- Document and metadata validation using Pydantic
- SHA-256 content integrity
- Source URL validation
- CRUD authorization
- Audit logging

Indexing:
- Data, chunking, embedding and indexing versions
- Incremental insert and update
- Stale chunk deletion
- Idempotency
- Scheme-level manifest

Observability:
- Langfuse spans
- Indexing operation metrics
- Operation status and audit events

The implementation uses isolated or existing Chroma collections.

In [24]:

import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager
from urllib.parse import urlparse

import casbin
from pydantic import BaseModel, ConfigDict, Field
from langchain_core.documents import Document

# ============================================================
# 1. CONFIGURATION AND VERSIONING
# ============================================================

INDEXING_VERSION_METADATA = {
    "data_version": "data_v1",
    "chunking_version": "chunk_v1",
    "embedding_version": "embed_v1",
    "indexing_version": "index_v1",
}

RETRIEVAL_CONFIG = {
    "retrieval_version": "retrieval_v1",
    "method": "hybrid_rrf",
    "top_k": 30,
}

ALLOWED_SOURCE_HOSTS = {
    "myscheme.gov.in",
    "www.myscheme.gov.in",
}

REQUIRE_SOURCE_URL = False

MANIFEST_PATH = Path("output/index_manifest.json")
AUDIT_PATH = Path("logs/indexing_audit.jsonl")

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. AUDIT LOGGING
# ============================================================

audit_logger = logging.getLogger("seva_indexing_audit")
audit_logger.setLevel(logging.INFO)
audit_logger.propagate = False

if not audit_logger.handlers:
    handler = logging.FileHandler(
        AUDIT_PATH,
        encoding="utf-8"
    )
    audit_logger.addHandler(handler)


def audit_event(actor, operation, scheme_id, status, details=None):
    event = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "actor": actor,
        "operation": operation,
        "scheme_id": scheme_id,
        "status": status,
        "details": details or {},
    }

    audit_logger.info(json.dumps(event, default=str))


# ============================================================
# 3. LANGFUSE OBSERVABILITY
# ============================================================

langfuse_client = None

try:
    from dotenv import load_dotenv
    from langfuse import get_client

    load_dotenv()
    langfuse_client = get_client()

except Exception as exc:
    print("Langfuse unavailable:", exc)


@contextmanager
def observe(name):
    if langfuse_client is not None:
        with langfuse_client.start_as_current_observation(
            name=name,
            as_type="span"
        ) as span:
            yield span
    else:
        yield None


# ============================================================
# 4. CASBIN RBAC AUTHORIZATION
# ============================================================

model = casbin.Model()

model.load_model_from_text("""
[request_definition]
r = sub, obj, act

[policy_definition]
p = sub, obj, act

[policy_effect]
e = some(where (p.eft == allow))

[matchers]
m = (p.sub == r.sub) && (p.obj == "*" || p.obj == r.obj) && (p.act == "*" || p.act == r.act)
""")

ENFORCER = casbin.Enforcer(model)

ENFORCER.add_policy("admin", "*", "*")

for operation in ["create", "read", "update", "delete"]:
    ENFORCER.add_policy(
        "ingestion_service", "*", operation
    )

ENFORCER.add_policy(
    "retrieval_service", "*", "read"
)


class AuthorizationError(Exception):
    pass


def authorize(role, collection, operation):
    if not ENFORCER.enforce(role, collection, operation):
        raise AuthorizationError(
            f"Access denied: {role} cannot {operation} "
            f"on {collection}"
        )


# ============================================================
# 5. PYDANTIC DOCUMENT VALIDATION
# ============================================================

class ChunkMetadata(BaseModel):

    model_config = ConfigDict(extra="allow")

    scheme_id: str = Field(min_length=1)
    chunk_id: str = Field(min_length=1)
    parent_id: str = Field(min_length=1)
    child_id: str = Field(min_length=1)
    source_field: str = Field(min_length=1)
    language: str = Field(min_length=1)

    data_version: str
    chunking_version: str
    embedding_version: str
    indexing_version: str

    source_url: str | None = None
    content_hash: str | None = None


def validate_source_url(metadata):

    source_url = metadata.get("source_url")

    if REQUIRE_SOURCE_URL and not source_url:
        raise ValueError("Source URL is required")

    if source_url:
        parsed = urlparse(source_url)

        if parsed.scheme != "https":
            raise ValueError("Only HTTPS source URLs are allowed")

        if parsed.hostname not in ALLOWED_SOURCE_HOSTS:
            raise ValueError(
                f"Untrusted source domain: {parsed.hostname}"
            )


def calculate_content_hash(doc):

    version_fields = set(INDEXING_VERSION_METADATA)

    excluded_fields = version_fields | {
        "content_hash",
        "indexed_at",
    }

    stable_metadata = {
        key: value
        for key, value in doc.metadata.items()
        if key not in excluded_fields
    }

    payload = {
        "content": doc.page_content,
        "metadata": stable_metadata,
    }

    serialized = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
        default=str,
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


def prepare_document(doc):

    if not doc.page_content or not doc.page_content.strip():
        raise ValueError("Empty document content")

    metadata = {
        **doc.metadata,
        **INDEXING_VERSION_METADATA,
    }

    validate_source_url(metadata)

    validated = ChunkMetadata.model_validate(metadata)

    prepared_doc = Document(
        page_content=doc.page_content,
        metadata=validated.model_dump(exclude_none=True),
    )

    prepared_doc.metadata["content_hash"] = (
        calculate_content_hash(prepared_doc)
    )

    return prepared_doc


# ============================================================
# 6. INDEX MANIFEST
# ============================================================

def load_manifest():

    if not MANIFEST_PATH.exists():
        return {}

    with open(MANIFEST_PATH, "r", encoding="utf-8") as file:
        return json.load(file)


def save_manifest(manifest):

    temp_path = MANIFEST_PATH.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)

    temp_path.replace(MANIFEST_PATH)


# ============================================================
# 7. EXISTING RECORD INSPECTION
# ============================================================

def get_scheme_records(db, scheme_id):

    return db._collection.get(
        where={"scheme_id": scheme_id},
        include=["metadatas"],
    )


# ============================================================
# 8. BACKFILL EXISTING VERSION METADATA
# ============================================================

def backfill_version_metadata(
    db,
    collection_name,
    batch_size=500,
    actor="admin",
):

    authorize(actor, collection_name, "update")

    collection = db._collection
    total = collection.count()
    updated = 0

    with observe("indexing.metadata_backfill") as span:

        for offset in range(0, total, batch_size):

            records = collection.get(
                limit=batch_size,
                offset=offset,
                include=["documents", "metadatas"],
            )

            ids = records["ids"]
            documents = records["documents"]
            metadatas = records["metadatas"]

            updated_metadata = []

            for content, metadata in zip(documents, metadatas):

                doc = Document(
                    page_content=content,
                    metadata={
                        **metadata,
                        **INDEXING_VERSION_METADATA,
                    },
                )

                validate_source_url(doc.metadata)

                validated = ChunkMetadata.model_validate(
                    doc.metadata
                )

                doc.metadata = validated.model_dump(
                    exclude_none=True
                )

                doc.metadata["content_hash"] = (
                    calculate_content_hash(doc)
                )

                updated_metadata.append(doc.metadata)

            if ids:
                # Metadata-only update: no embedding regeneration.
                collection.update(
                    ids=ids,
                    metadatas=updated_metadata,
                )

            updated += len(ids)

        if span:
            span.update(metadata={
                "collection": collection_name,
                "documents_updated": updated,
                "batch_size": batch_size,
            })

    audit_event(
        actor,
        "metadata_backfill",
        "ALL",
        "completed",
        {"collection": collection_name, "updated": updated},
    )

    return updated


# ============================================================
# 9. INCREMENTAL COLLECTION INDEXING
# ============================================================

def index_collection(
    db,
    documents,
    scheme_id,
    collection_name,
    actor,
):

    collection = db._collection

    existing = get_scheme_records(db, scheme_id)

    existing_ids = set(existing["ids"])

    existing_metadata = dict(zip(
        existing["ids"],
        existing["metadatas"],
    ))

    prepared_docs = [
        prepare_document(doc)
        for doc in documents
    ]

    incoming_ids = [
        doc.metadata["chunk_id"]
        for doc in prepared_docs
    ]

    if len(incoming_ids) != len(set(incoming_ids)):
        raise ValueError("Duplicate chunk IDs in input")

    for doc in prepared_docs:
        if doc.metadata["scheme_id"] != scheme_id:
            raise ValueError("Scheme ID mismatch")

    new_docs = []
    changed_docs = []
    skipped = 0

    for doc in prepared_docs:

        chunk_id = doc.metadata["chunk_id"]
        old_metadata = existing_metadata.get(chunk_id)

        if old_metadata is None:
            new_docs.append(doc)

        elif (
            old_metadata.get("content_hash")
            == doc.metadata["content_hash"]
            and all(
                old_metadata.get(key) == value
                for key, value in INDEXING_VERSION_METADATA.items()
            )
        ):
            skipped += 1

        else:
            changed_docs.append(doc)

    stale_ids = existing_ids - set(incoming_ids)

    if new_docs:
        authorize(actor, collection_name, "create")

    if changed_docs:
        authorize(actor, collection_name, "update")

    if stale_ids:
        authorize(actor, collection_name, "delete")

    with observe("indexing.collection_upsert") as span:

        # LangChain Chroma add_documents uses upsert semantics.
        # Only changed and new chunks are embedded.

        if new_docs:
            db.add_documents(
                documents=new_docs,
                ids=[
                    doc.metadata["chunk_id"]
                    for doc in new_docs
                ],
            )

        if changed_docs:
            db.add_documents(
                documents=changed_docs,
                ids=[
                    doc.metadata["chunk_id"]
                    for doc in changed_docs
                ],
            )

        if stale_ids:
            db.delete(ids=list(stale_ids))

        metrics = {
            "collection": collection_name,
            "inserted": len(new_docs),
            "updated": len(changed_docs),
            "deleted": len(stale_ids),
            "skipped": skipped,
        }

        if span:
            span.update(metadata=metrics)

    return metrics


# ============================================================
# 10. SECURE SCHEME INSERT / UPDATE
# ============================================================

def secure_index_scheme(
    scheme_id,
    discovery_documents,
    detail_documents,
    actor="ingestion_service",
):

    discovery_existing = get_scheme_records(
        discovery_db, scheme_id
    )

    detail_existing = get_scheme_records(
        detail_db, scheme_id
    )

    is_new = not (
        discovery_existing["ids"] or detail_existing["ids"]
    )

    operation = "create" if is_new else "update"

    authorize(actor, "discovery", operation)
    authorize(actor, "detail", operation)

    audit_event(
        actor, operation, scheme_id, "started"
    )

    try:

        with observe("indexing.scheme") as span:

            discovery_metrics = index_collection(
                discovery_db,
                discovery_documents,
                scheme_id,
                "discovery",
                actor,
            )

            detail_metrics = index_collection(
                detail_db,
                detail_documents,
                scheme_id,
                "detail",
                actor,
            )

            manifest = load_manifest()

            manifest[scheme_id] = {
                "scheme_id": scheme_id,
                "versions": INDEXING_VERSION_METADATA,
                "discovery_chunk_ids": [
                    doc.metadata["chunk_id"]
                    for doc in discovery_documents
                ],
                "detail_chunk_ids": [
                    doc.metadata["chunk_id"]
                    for doc in detail_documents
                ],
                "index_status": "completed",
                "indexed_at": datetime.now(
                    timezone.utc
                ).isoformat(),
            }

            save_manifest(manifest)

            result = {
                "scheme_id": scheme_id,
                "operation": operation,
                "discovery": discovery_metrics,
                "detail": detail_metrics,
            }

            if span:
                span.update(metadata=result)

        audit_event(
            actor, operation, scheme_id, "completed", result
        )

        return result

    except Exception as exc:

        audit_event(
            actor,
            operation,
            scheme_id,
            "failed",
            {"error": str(exc)},
        )

        raise


# ============================================================
# 11. SECURE SCHEME READ
# ============================================================

def secure_read_scheme(
    scheme_id,
    actor="retrieval_service",
):

    authorize(actor, "discovery", "read")
    authorize(actor, "detail", "read")

    discovery = get_scheme_records(
        discovery_db, scheme_id
    )

    detail = get_scheme_records(
        detail_db, scheme_id
    )

    return {
        "discovery": [
            Document(page_content="", metadata=metadata)
            for metadata in discovery["metadatas"]
        ],
        "detail": [
            Document(page_content="", metadata=metadata)
            for metadata in detail["metadatas"]
        ],
        "discovery_count": len(discovery["ids"]),
        "detail_count": len(detail["ids"]),
    }


# ============================================================
# 12. SECURE VECTOR RETRIEVAL
# ============================================================

def secure_retrieve(
    db,
    query,
    collection_name,
    actor="retrieval_service",
    k=5,
):

    authorize(actor, collection_name, "read")

    with observe("retrieval.secure_search") as span:

        results = db.similarity_search_with_score(
            query=query,
            k=k,
        )

        if span:
            span.update(metadata={
                "collection": collection_name,
                "retrieval_version": RETRIEVAL_CONFIG[
                    "retrieval_version"
                ],
                "top_k": k,
                "result_count": len(results),
            })

    return results


# ============================================================
# 13. SECURE CASCADE DELETE
# ============================================================

def secure_delete_scheme(
    scheme_id,
    actor="ingestion_service",
    cache_invalidator=None,
):

    authorize(actor, "discovery", "delete")
    authorize(actor, "detail", "delete")

    audit_event(
        actor, "delete", scheme_id, "started"
    )

    try:

        with observe("indexing.scheme_delete") as span:

            discovery_before = len(
                get_scheme_records(discovery_db, scheme_id)["ids"]
            )

            detail_before = len(
                get_scheme_records(detail_db, scheme_id)["ids"]
            )

            discovery_db.delete(
                where={"scheme_id": scheme_id}
            )

            detail_db.delete(
                where={"scheme_id": scheme_id}
            )

            # Invalidate application cache if configured.
            if cache_invalidator:
                cache_invalidator(scheme_id)

            manifest = load_manifest()
            manifest.pop(scheme_id, None)
            save_manifest(manifest)

            result = {
                "scheme_id": scheme_id,
                "discovery_deleted": discovery_before,
                "detail_deleted": detail_before,
                "total_deleted": (
                    discovery_before + detail_before
                ),
            }

            if span:
                span.update(metadata=result)

        audit_event(
            actor, "delete", scheme_id, "completed", result
        )

        return result

    except Exception as exc:

        audit_event(
            actor,
            "delete",
            scheme_id,
            "failed",
            {"error": str(exc)},
        )

        raise


# ============================================================
# 14. FINAL OBSERVABILITY FLUSH
# ============================================================

def flush_observability():

    if langfuse_client is not None:
        langfuse_client.flush()

    print("Observability flush requested.")


In [25]:
discovery_updated = backfill_version_metadata(
    discovery_db, "discovery"
)

detail_updated = backfill_version_metadata(
    detail_db, "detail"
)

print(discovery_updated, detail_updated)

flush_observability()

23438 67945
Observability flush requested.


## Verify Version Metadata Backfill

### Purpose
Verify that all existing Discovery and Detail records contain the expected version metadata without changing their vector IDs or embeddings.

In [27]:
VERSION_FIELDS = {
    "data_version": "data_v1",
    "chunking_version": "chunk_v1",
    "embedding_version": "embed_v1",
    "indexing_version": "index_v1",
}

def verify_version_metadata(db, collection_name):
    collection = db._collection
    total = collection.count()
    valid = 0
    invalid = 0

    for offset in range(0, total, 500):
        records = collection.get(
            limit=500,
            offset=offset,
            include=["metadatas"]
        )

        for metadata in records["metadatas"]:
            if all(
                metadata.get(key) == value
                for key, value in VERSION_FIELDS.items()
            ):
                valid += 1
            else:
                invalid += 1

    return {
        "collection": collection_name,
        "total": total,
        "valid": valid,
        "invalid": invalid,
    }


discovery_result = verify_version_metadata(
    discovery_db, "discovery"
)

detail_result = verify_version_metadata(
    detail_db, "detail"
)

print(discovery_result)
print(detail_result)

assert discovery_result["invalid"] == 0
assert detail_result["invalid"] == 0

print("\nAll version metadata verified!")

{'collection': 'discovery', 'total': 23438, 'valid': 23438, 'invalid': 0}
{'collection': 'detail', 'total': 67945, 'valid': 67945, 'invalid': 0}

All version metadata verified!


## End-to-End Secure Incremental Indexing with Langfuse

### Purpose

Evaluate the complete secure indexing lifecycle using isolated ChromaDB collections.

Operations:
- INSERT
- READ
- IDEMPOTENCY
- UPDATE
- VERSION VALIDATION
- UNAUTHORIZED DELETE
- CASCADE DELETE

Observability:
- Individual parent span for every operation
- Nested spans for internal indexing operations
- Automatic latency measurement
- Operation metrics and status metadata

All tests use synthetic data and temporary collections.

In [28]:

import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document
from datetime import datetime, timezone
import traceback


# ============================================================
# 1. ISOLATED TEST ENVIRONMENT
# ============================================================

original_discovery_db = discovery_db
original_detail_db = detail_db
original_manifest_path = MANIFEST_PATH

test_client = chromadb.EphemeralClient()

test_discovery_db = Chroma(
    client=test_client,
    collection_name="seva_e2e_discovery_test",
    embedding_function=original_discovery_db._embedding_function,
    collection_metadata={"hnsw:space": "l2"},
)

test_detail_db = Chroma(
    client=test_client,
    collection_name="seva_e2e_detail_test",
    embedding_function=original_detail_db._embedding_function,
    collection_metadata={"hnsw:space": "l2"},
)

discovery_db = test_discovery_db
detail_db = test_detail_db

MANIFEST_PATH = Path("output/test_index_manifest.json")

SCHEME_ID = "CRUD_E2E_TEST_001"

test_results = []


# ============================================================
# 2. LANGFUSE OPERATION SPAN
# ============================================================

def run_observed_operation(operation_name, callback):

    started_at = datetime.now(timezone.utc)

    with observe(f"experiment.crud.{operation_name}") as span:

        try:
            result = callback()

            if span:
                span.update(
                    level="DEFAULT",
                    status_message="Operation completed",
                    metadata={
                        "operation": operation_name,
                        "scheme_id": SCHEME_ID,
                        "status": "success",
                        "result": result if isinstance(result, dict)
                        else str(result),
                    },
                )

            test_results.append({
                "operation": operation_name,
                "status": "success",
                "started_at": started_at.isoformat(),
            })

            print(f"{operation_name.upper()}: PASSED")

            return result

        except Exception as exc:

            if span:
                span.update(
                    level="ERROR",
                    status_message=str(exc),
                    metadata={
                        "operation": operation_name,
                        "scheme_id": SCHEME_ID,
                        "status": "failed",
                    },
                )

            test_results.append({
                "operation": operation_name,
                "status": "failed",
                "error": str(exc),
            })

            raise


# ============================================================
# 3. SYNTHETIC DOCUMENT FACTORY
# ============================================================

def make_test_doc(chunk_id, source_field, content):

    return Document(
        page_content=content,
        metadata={
            "scheme_id": SCHEME_ID,
            "chunk_id": chunk_id,
            "parent_id": f"{SCHEME_ID}_{source_field}",
            "child_id": f"{SCHEME_ID}_{source_field}",
            "source_field": source_field,
            "language": "en",
        },
    )


discovery_v1 = [
    make_test_doc(
        f"{SCHEME_ID}_OVERVIEW_001",
        "overview",
        "Government scheme overview version one.",
    ),
    make_test_doc(
        f"{SCHEME_ID}_BENEFITS_001",
        "benefits",
        "Scheme benefits version one.",
    ),
]

detail_v1 = [
    make_test_doc(
        f"{SCHEME_ID}_FAQ_001",
        "faq",
        "Who can apply for this scheme?",
    ),
    make_test_doc(
        f"{SCHEME_ID}_DOCUMENTS_001",
        "documents_required",
        "Aadhaar card and income certificate.",
    ),
]


# ============================================================
# 4. INSERT
# ============================================================

def test_insert():

    result = secure_index_scheme(
        scheme_id=SCHEME_ID,
        discovery_documents=discovery_v1,
        detail_documents=detail_v1,
        actor="ingestion_service",
    )

    assert result["discovery"]["inserted"] == 2
    assert result["detail"]["inserted"] == 2

    return result


insert_result = run_observed_operation(
    "insert",
    test_insert,
)


# ============================================================
# 5. READ
# ============================================================

def test_read():

    result = secure_read_scheme(
        scheme_id=SCHEME_ID,
        actor="retrieval_service",
    )

    assert result["discovery_count"] == 2
    assert result["detail_count"] == 2

    return {
        "discovery_count": result["discovery_count"],
        "detail_count": result["detail_count"],
    }


read_result = run_observed_operation(
    "read",
    test_read,
)


# ============================================================
# 6. IDEMPOTENCY
# ============================================================

def test_idempotency():

    result = secure_index_scheme(
        scheme_id=SCHEME_ID,
        discovery_documents=discovery_v1,
        detail_documents=detail_v1,
        actor="ingestion_service",
    )

    assert result["discovery"]["inserted"] == 0
    assert result["discovery"]["updated"] == 0
    assert result["discovery"]["skipped"] == 2

    assert result["detail"]["inserted"] == 0
    assert result["detail"]["updated"] == 0
    assert result["detail"]["skipped"] == 2

    return result


idempotency_result = run_observed_operation(
    "idempotency",
    test_idempotency,
)


# ============================================================
# 7. UPDATE
# ============================================================

discovery_v2 = [
    make_test_doc(
        f"{SCHEME_ID}_OVERVIEW_001",
        "overview",
        "Government scheme overview UPDATED version two.",
    ),
    make_test_doc(
        f"{SCHEME_ID}_BENEFITS_001",
        "benefits",
        "Scheme benefits version one.",
    ),
    make_test_doc(
        f"{SCHEME_ID}_ELIGIBILITY_001",
        "eligibility",
        "Applicants must meet eligibility criteria.",
    ),
]

detail_v2 = [
    make_test_doc(
        f"{SCHEME_ID}_FAQ_001",
        "faq",
        "Who can apply for this scheme?",
    ),
    make_test_doc(
        f"{SCHEME_ID}_APPLICATION_001",
        "application_process",
        "Apply through the official government portal.",
    ),
]


def test_update():

    result = secure_index_scheme(
        scheme_id=SCHEME_ID,
        discovery_documents=discovery_v2,
        detail_documents=detail_v2,
        actor="ingestion_service",
    )

    assert result["discovery"]["updated"] == 1
    assert result["discovery"]["inserted"] == 1
    assert result["discovery"]["skipped"] == 1

    assert result["detail"]["inserted"] == 1
    assert result["detail"]["deleted"] == 1
    assert result["detail"]["skipped"] == 1

    return result


update_result = run_observed_operation(
    "update",
    test_update,
)


# ============================================================
# 8. VERSION METADATA VALIDATION
# ============================================================

def test_version_metadata():

    checked = 0

    for db in [discovery_db, detail_db]:

        records = db._collection.get(
            where={"scheme_id": SCHEME_ID},
            include=["metadatas"],
        )

        for metadata in records["metadatas"]:

            for key, value in INDEXING_VERSION_METADATA.items():
                assert metadata.get(key) == value

            assert metadata.get("content_hash")

            checked += 1

    return {"documents_verified": checked}


version_result = run_observed_operation(
    "version_validation",
    test_version_metadata,
)


# ============================================================
# 9. UNAUTHORIZED DELETE
# ============================================================

def test_unauthorized_delete():

    try:

        secure_delete_scheme(
            scheme_id=SCHEME_ID,
            actor="retrieval_service",
        )

        raise AssertionError(
            "Unauthorized deletion was allowed"
        )

    except AuthorizationError:

        return {"unauthorized_access": "rejected"}


unauthorized_result = run_observed_operation(
    "unauthorized_delete",
    test_unauthorized_delete,
)


# ============================================================
# 10. CASCADE DELETE
# ============================================================

def test_cascade_delete():

    result = secure_delete_scheme(
        scheme_id=SCHEME_ID,
        actor="ingestion_service",
    )

    assert result["discovery_deleted"] == 3
    assert result["detail_deleted"] == 2

    assert discovery_db._collection.count() == 0
    assert detail_db._collection.count() == 0

    manifest = load_manifest()
    assert SCHEME_ID not in manifest

    return result


delete_result = run_observed_operation(
    "cascade_delete",
    test_cascade_delete,
)


# ============================================================
# 11. FINAL REPORT
# ============================================================

print("\n" + "=" * 55)
print("SEVA_AI SECURE CRUD TEST REPORT")
print("=" * 55)

for item in test_results:
    print(item)

assert all(
    item["status"] == "success"
    for item in test_results
)

print("\nALL TESTS PASSED!")

print("\nFinal Discovery count:",
      discovery_db._collection.count())

print("Final Detail count:",
      detail_db._collection.count())


# ============================================================
# 12. RESTORE ORIGINAL COLLECTION REFERENCES
# ============================================================

discovery_db = original_discovery_db
detail_db = original_detail_db
MANIFEST_PATH = original_manifest_path


# ============================================================
# 13. EXPORT LANGFUSE TRACES
# ============================================================

flush_observability()


INSERT: PASSED
READ: PASSED
IDEMPOTENCY: PASSED
UPDATE: PASSED
VERSION_VALIDATION: PASSED
UNAUTHORIZED_DELETE: PASSED
CASCADE_DELETE: PASSED

SEVA_AI SECURE CRUD TEST REPORT
{'operation': 'insert', 'status': 'success', 'started_at': '2026-11-22T05:23:19.178391+00:00'}
{'operation': 'read', 'status': 'success', 'started_at': '2026-11-22T05:23:20.211148+00:00'}
{'operation': 'idempotency', 'status': 'success', 'started_at': '2026-11-22T05:23:20.229148+00:00'}
{'operation': 'update', 'status': 'success', 'started_at': '2026-11-22T05:23:20.351144+00:00'}
{'operation': 'version_validation', 'status': 'success', 'started_at': '2026-11-22T05:23:21.066124+00:00'}
{'operation': 'unauthorized_delete', 'status': 'success', 'started_at': '2026-11-22T05:23:21.082131+00:00'}
{'operation': 'cascade_delete', 'status': 'success', 'started_at': '2026-11-22T05:23:21.088125+00:00'}

ALL TESTS PASSED!

Final Discovery count: 0
Final Detail count: 0
Observability flush requested.


## Failure Recovery and Consistency Testing

### Purpose

Test recovery from partial indexing failures across Discovery and Detail collections.

Scenarios:
1. Start a scheme indexing operation.
2. Successfully index Discovery.
3. Simulate failure during Detail indexing.
4. Persist operation status and checkpoints.
5. Retry the failed operation.
6. Skip already completed Discovery indexing.
7. Complete Detail indexing.
8. Update the manifest only after both collections succeed.
9. Verify final collection consistency.

Use an operation journal to support retry and recovery.

In [ ]:

import json
import hashlib
import chromadb

from pathlib import Path
from datetime import datetime, timezone
from langchain_chroma import Chroma
from langchain_core.documents import Document


# ============================================================
# 1. ISOLATED TEST COLLECTIONS
# ============================================================

original_discovery_db = discovery_db
original_detail_db = detail_db
original_manifest_path = MANIFEST_PATH
original_index_collection = index_collection

test_client = chromadb.EphemeralClient()

discovery_db = Chroma(
    client=test_client,
    collection_name="seva_recovery_discovery_test",
    embedding_function=original_discovery_db._embedding_function,
    collection_metadata={"hnsw:space": "l2"},
)

detail_db = Chroma(
    client=test_client,
    collection_name="seva_recovery_detail_test",
    embedding_function=original_detail_db._embedding_function,
    collection_metadata={"hnsw:space": "l2"},
)

MANIFEST_PATH = Path("output/recovery_test_manifest.json")

JOURNAL_PATH = Path("output/indexing_operation_journal.json")
JOURNAL_PATH.parent.mkdir(parents=True, exist_ok=True)

SCHEME_ID = "RECOVERY_TEST_001"


# ============================================================
# 2. ATOMIC JSON JOURNAL STORAGE
# ============================================================

def load_operation_journal():

    if not JOURNAL_PATH.exists():
        return {}

    with open(JOURNAL_PATH, "r", encoding="utf-8") as file:
        return json.load(file)


def save_operation_journal(journal):

    temp_path = JOURNAL_PATH.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(journal, file, indent=2)

    temp_path.replace(JOURNAL_PATH)


# ============================================================
# 3. GENERATE DETERMINISTIC OPERATION ID
# ============================================================

def generate_operation_id(
    scheme_id,
    discovery_documents,
    detail_documents,
):

    prepared = [
        prepare_document(doc)
        for doc in discovery_documents + detail_documents
    ]

    payload = {
        "scheme_id": scheme_id,
        "versions": INDEXING_VERSION_METADATA,
        "chunks": sorted([
            {
                "chunk_id": doc.metadata["chunk_id"],
                "content_hash": doc.metadata["content_hash"],
            }
            for doc in prepared
        ], key=lambda item: item["chunk_id"]),
    }

    serialized = json.dumps(
        payload,
        sort_keys=True,
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


# ============================================================
# 4. RECOVERABLE INDEXING ORCHESTRATOR
# ============================================================

def recoverable_index_scheme(
    scheme_id,
    discovery_documents,
    detail_documents,
    actor="ingestion_service",
):

    authorize(actor, "discovery", "create")
    authorize(actor, "detail", "create")

    operation_id = generate_operation_id(
        scheme_id,
        discovery_documents,
        detail_documents,
    )

    journal = load_operation_journal()

    operation = journal.get(operation_id)

    if operation and operation["status"] == "completed":

        print("Operation already completed. Skipping.")

        return {
            "operation_id": operation_id,
            "status": "already_completed",
        }

    if operation is None:

        operation = {
            "operation_id": operation_id,
            "scheme_id": scheme_id,
            "status": "pending",
            "discovery_completed": False,
            "detail_completed": False,
            "created_at": datetime.now(
                timezone.utc
            ).isoformat(),
            "checkpoints": {},
        }

        journal[operation_id] = operation
        save_operation_journal(journal)

    operation["status"] = "in_progress"
    operation["updated_at"] = datetime.now(
        timezone.utc
    ).isoformat()

    save_operation_journal(journal)

    try:

        with observe("indexing.recovery.scheme") as parent_span:

            # ------------------------------------------------
            # DISCOVERY CHECKPOINT
            # ------------------------------------------------

            if not operation["discovery_completed"]:

                authorize(actor, "discovery", "create")

                with observe(
                    "indexing.recovery.discovery"
                ) as child_span:

                    discovery_metrics = index_collection(
                        discovery_db,
                        discovery_documents,
                        scheme_id,
                        "discovery",
                        actor,
                    )

                    operation["checkpoints"]["discovery"] = (
                        discovery_metrics
                    )

                    operation["discovery_completed"] = True

                    journal[operation_id] = operation
                    save_operation_journal(journal)

                    if child_span:
                        child_span.update(metadata={
                            "status": "completed",
                            "metrics": discovery_metrics,
                        })

            else:

                print("Discovery checkpoint already completed.")

            # ------------------------------------------------
            # DETAIL CHECKPOINT
            # ------------------------------------------------

            if not operation["detail_completed"]:

                authorize(actor, "detail", "create")

                with observe(
                    "indexing.recovery.detail"
                ) as child_span:

                    detail_metrics = index_collection(
                        detail_db,
                        detail_documents,
                        scheme_id,
                        "detail",
                        actor,
                    )

                    operation["checkpoints"]["detail"] = (
                        detail_metrics
                    )

                    operation["detail_completed"] = True

                    journal[operation_id] = operation
                    save_operation_journal(journal)

                    if child_span:
                        child_span.update(metadata={
                            "status": "completed",
                            "metrics": detail_metrics,
                        })

            else:

                print("Detail checkpoint already completed.")

            # ------------------------------------------------
            # FINALIZE MANIFEST ONLY AFTER BOTH SUCCEED
            # ------------------------------------------------

            if (
                operation["discovery_completed"]
                and operation["detail_completed"]
            ):

                manifest = load_manifest()

                manifest[scheme_id] = {
                    "scheme_id": scheme_id,
                    "operation_id": operation_id,
                    "versions": INDEXING_VERSION_METADATA,
                    "discovery_chunk_ids": [
                        doc.metadata["chunk_id"]
                        for doc in discovery_documents
                    ],
                    "detail_chunk_ids": [
                        doc.metadata["chunk_id"]
                        for doc in detail_documents
                    ],
                    "index_status": "completed",
                    "indexed_at": datetime.now(
                        timezone.utc
                    ).isoformat(),
                }

                save_manifest(manifest)

                operation["status"] = "completed"
                operation["completed_at"] = datetime.now(
                    timezone.utc
                ).isoformat()

                journal[operation_id] = operation
                save_operation_journal(journal)

            if parent_span:
                parent_span.update(metadata={
                    "scheme_id": scheme_id,
                    "operation_id": operation_id,
                    "status": operation["status"],
                })

        audit_event(
            actor,
            "recoverable_indexing",
            scheme_id,
            operation["status"],
            {"operation_id": operation_id},
        )

        return {
            "operation_id": operation_id,
            "status": operation["status"],
            "discovery_completed": operation[
                "discovery_completed"
            ],
            "detail_completed": operation[
                "detail_completed"
            ],
        }

    except Exception as exc:

        operation["status"] = "failed"
        operation["last_error"] = str(exc)
        operation["failed_at"] = datetime.now(
            timezone.utc
        ).isoformat()

        journal[operation_id] = operation
        save_operation_journal(journal)

        audit_event(
            actor,
            "recoverable_indexing",
            scheme_id,
            "failed",
            {
                "operation_id": operation_id,
                "error": str(exc),
            },
        )

        raise


# ============================================================
# 5. CREATE SYNTHETIC SCHEME
# ============================================================

discovery_docs = [
    Document(
        page_content="Recovery test scheme overview.",
        metadata={
            "scheme_id": SCHEME_ID,
            "chunk_id": f"{SCHEME_ID}_OVERVIEW_001",
            "parent_id": f"{SCHEME_ID}_OVERVIEW",
            "child_id": f"{SCHEME_ID}_OVERVIEW",
            "source_field": "overview",
            "language": "en",
        },
    ),
    Document(
        page_content="Recovery test scheme benefits.",
        metadata={
            "scheme_id": SCHEME_ID,
            "chunk_id": f"{SCHEME_ID}_BENEFITS_001",
            "parent_id": f"{SCHEME_ID}_BENEFITS",
            "child_id": f"{SCHEME_ID}_BENEFITS",
            "source_field": "benefits",
            "language": "en",
        },
    ),
]

detail_docs = [
    Document(
        page_content="Recovery test eligibility criteria.",
        metadata={
            "scheme_id": SCHEME_ID,
            "chunk_id": f"{SCHEME_ID}_ELIGIBILITY_001",
            "parent_id": f"{SCHEME_ID}_ELIGIBILITY",
            "child_id": f"{SCHEME_ID}_ELIGIBILITY",
            "source_field": "eligibility",
            "language": "en",
        },
    ),
    Document(
        page_content="Recovery test application process.",
        metadata={
            "scheme_id": SCHEME_ID,
            "chunk_id": f"{SCHEME_ID}_APPLICATION_001",
            "parent_id": f"{SCHEME_ID}_APPLICATION",
            "child_id": f"{SCHEME_ID}_APPLICATION",
            "source_field": "application_process",
            "language": "en",
        },
    ),
]

# ============================================================
# 6. SIMULATE FAILURE DURING DETAIL INDEXING
# ============================================================

detail_failure_triggered = False

def simulate_detail_failure(
    db,
    documents,
    scheme_id,
    collection_name,
    actor,
):

    global detail_failure_triggered

    if (
        collection_name == "detail"
        and not detail_failure_triggered
    ):

        detail_failure_triggered = True

        raise RuntimeError(
            "Simulated failure during Detail indexing"
        )

    return original_index_collection(
        db,
        documents,
        scheme_id,
        collection_name,
        actor,
    )

index_collection = simulate_detail_failure

# ============================================================
# 7. FIRST ATTEMPT: EXPECTED PARTIAL FAILURE
# ============================================================

operation_id = generate_operation_id(
    SCHEME_ID,
    discovery_docs,
    detail_docs,
)

try:

    recoverable_index_scheme(
        SCHEME_ID,
        discovery_docs,
        detail_docs,
    )

    raise AssertionError(
        "Expected simulated Detail failure"
    )

except RuntimeError as exc:

    assert "Simulated failure" in str(exc)

    print("Simulated failure detected.")

journal = load_operation_journal()
failed_operation = journal[operation_id]

assert failed_operation["status"] == "failed"
assert failed_operation["discovery_completed"] is True
assert failed_operation["detail_completed"] is False

assert discovery_db._collection.count() == 2
assert detail_db._collection.count() == 0

print("Partial failure recorded successfully.")


# ============================================================
# 8. SECOND ATTEMPT: RECOVERY
# ============================================================

index_collection = original_index_collection

recovery_result = recoverable_index_scheme(
    SCHEME_ID,
    discovery_docs,
    detail_docs,
)

assert recovery_result["status"] == "completed"

assert discovery_db._collection.count() == 2
assert detail_db._collection.count() == 2

print("Recovery completed successfully.")


# ============================================================
# 9. VERIFY MANIFEST AND JOURNAL
# ============================================================

final_journal = load_operation_journal()
final_operation = final_journal[operation_id]

final_manifest = load_manifest()

assert final_operation["status"] == "completed"
assert final_operation["discovery_completed"] is True
assert final_operation["detail_completed"] is True

assert final_manifest[SCHEME_ID]["index_status"] == "completed"

assert discovery_db._collection.count() == 2
assert detail_db._collection.count() == 2

print("\nFINAL RECOVERY REPORT")
print("Discovery chunks:", discovery_db._collection.count())
print("Detail chunks:", detail_db._collection.count())
print("Operation status:", final_operation["status"])
print("Manifest status:", final_manifest[SCHEME_ID]["index_status"])

print("\nALL FAILURE RECOVERY TESTS PASSED!")


# ============================================================
# 10. CLEANUP AND RESTORE ORIGINAL REFERENCES
# ============================================================

finally_cleanup = False

# Remove the synthetic scheme from temporary collections.
discovery_db.delete(where={"scheme_id": SCHEME_ID})
detail_db.delete(where={"scheme_id": SCHEME_ID})

manifest = load_manifest()
manifest.pop(SCHEME_ID, None)
save_manifest(manifest)

discovery_db = original_discovery_db
detail_db = original_detail_db
MANIFEST_PATH = original_manifest_path
index_collection = original_index_collection

flush_observability()


Simulated failure detected.
Partial failure recorded successfully.
Discovery checkpoint already completed.
Recovery completed successfully.

FINAL RECOVERY REPORT
Discovery chunks: 2
Detail chunks: 2
Operation status: completed
Manifest status: completed

ALL FAILURE RECOVERY TESTS PASSED!
Observability flush requested.
